# ⚡ EnerVision AI — Fair 24-Hour Comparison (XGBoost vs. PatchTSMixer)
**Why this notebook exists:**

XGBoost and Bi-LSTM were only ever evaluated on next-minute (single-step) accuracy, while PatchTSMixer was evaluated on a full 24-hour forecast. Comparing their headline numbers directly was unfair — like comparing scores from an easy test and a hard test.

This notebook fixes that: it runs XGBoost using its actual deployment method (recursive forecasting) so it predicts a full 24 hours ahead, just like PatchTSMixer, then compares both models on the exact same task.

**Note:** Bi-LSTM is not included in this comparison because it has no existing mechanism for hourly forecasting (it was trained on minute-level windows only). Its earlier single-step number (0.2137) is kept for reference only, not for comparison.

## 📦 Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import PatchTSMixerConfig, PatchTSMixerForPrediction

pd.set_option('display.width', 120)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 📂 Step 2: Load the Cleaned (per-minute) Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/EnerVision_AI/Data/HomecCleaned.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

print(f"✅ Loaded {df.shape[0]} minute-level rows, from {df['time'].min()} to {df['time'].max()}")


## 🗓️ Step 3: Define a Shared Test PeriodBoth models
 must be evaluated on **exactly the same calendar time window**, otherwise"fair" comparison is undermined by the two models' different native resolutions(per-minute for XGBoost, hourly for PatchTSMixer) producing different row-count-basedsplits. We define the test period as the **last 20% of the full time span**, by calendartime — matching the 80/20 split ratio used when training all three models — and requireeach anchor point to have at least 24 hours of future actual data (to score against) and24 hours of past actual data (for the lag-1440 / same-hour-yesterday weather proxy).

In [ ]:
total_span = df['time'].max() - df['time'].min()
TEST_START_TIME = df['time'].min() + total_span * 0.8

# Valid anchors: at least 24h of history before them, and at least 24h of future data after them
valid_start = max(TEST_START_TIME, df['time'].min() + pd.Timedelta(hours=24))
valid_end = df['time'].max() - pd.Timedelta(hours=24)

# Ensure valid_start and valid_end are aligned to the hour for consistency with df_hourly
valid_start = valid_start.ceil('h')
valid_end = valid_end.floor('h')

print(f"Test period starts: {TEST_START_TIME}")
print(f"Valid anchor range (hourly aligned): {valid_start} -> {valid_end}")

## 🕐 Step 4: Build the Hourly Dataset
(for PatchTSMixer and for ground-truth comparison)Same construction used when retraining PatchTSMixer: target + weather resampled to hourlymeans, calendar features recomputed directly at hourly resolution.

In [ ]:
TARGET_COL = 'use [kW]'  # household Demand

WEATHER_COLS = ['temperature', 'humidity', 'visibility', 'apparentTemperature',
                'pressure', 'windSpeed', 'cloudCover', 'windBearing',
                'precipIntensity', 'dewPoint', 'precipProbability']
WEATHER_COLS = [c for c in WEATHER_COLS if c in df.columns]

agg_dict = {TARGET_COL: 'mean'}
agg_dict.update({c: 'mean' for c in WEATHER_COLS})

df_hourly = (
    df.set_index('time')
      .resample('h')
      .agg(agg_dict)
      .dropna()
      .reset_index()
)

df_hourly['is_weekend'] = (df_hourly['time'].dt.dayofweek >= 5).astype(float)
df_hourly['hour_sin'] = np.sin(2 * np.pi * df_hourly['time'].dt.hour / 24.0)
df_hourly['hour_cos'] = np.cos(2 * np.pi * df_hourly['time'].dt.hour / 24.0)
df_hourly['month_sin'] = np.sin(2 * np.pi * df_hourly['time'].dt.month / 12.0)
df_hourly['month_cos'] = np.cos(2 * np.pi * df_hourly['time'].dt.month / 12.0)
CALENDAR_COLS = ['is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']

PATCH_FEATURE_COLS = [TARGET_COL] + WEATHER_COLS + CALENDAR_COLS
TARGET_CHANNEL_IDX = 0

print(f"Hourly dataset: {df_hourly.shape[0]} rows, {len(PATCH_FEATURE_COLS)} channels")
df_hourly = df_hourly.set_index('time')


## 🧠 Step 5: Load the Trained Models

In [ ]:
# --- XGBoost (full pipeline: preprocessing + model) ---
XGB_PATH = '/content/drive/MyDrive/EnerVision_AI/Models/XGBoost/xgboost_Modele.pkl'
xgb_pipeline = joblib.load(XGB_PATH)
print("✅ XGBoost pipeline loaded.")

# --- PatchTSMixer (weights + scaler; config must match training exactly) ---
PATCH_MODEL_PATH = '/content/drive/MyDrive/EnerVision_AI/Models/PatchTSMixer/patchtsmixer_model.pt'
PATCH_SCALER_PATH = '/content/drive/MyDrive/EnerVision_AI/Models/PatchTSMixer/patchtsmixer_scaler.joblib'

CONTEXT_LENGTH = 96
PREDICTION_LENGTH = 24

patch_config = PatchTSMixerConfig(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    num_input_channels=len(PATCH_FEATURE_COLS),
    prediction_channel_indices=[TARGET_CHANNEL_IDX],
    patch_length=16,
    patch_stride=8,
    d_model=64,
    num_layers=3,
    dropout=0.2,
)
patch_model = PatchTSMixerForPrediction(patch_config).to(device)
patch_model.load_state_dict(torch.load(PATCH_MODEL_PATH, map_location=device))
patch_model.eval()

patch_scaler = joblib.load(PATCH_SCALER_PATH)
print("✅ PatchTSMixer model + scaler loaded.")


## 🔁 Step 6: XGBoost Recursive 24-Hour ForecastReplicates
 `connected.py`'s `predict_xgb_recursive_isolated` exactly, generalized to runfrom any anchor timestamp (not just a single demo point).

In [ ]:
XGB_FEATURES = [
    "temperature", "humidity", "visibility", "apparentTemperature", "pressure",
    "windSpeed", "cloudCover", "windBearing", "precipIntensity", "dewPoint",
    "precipProbability", "is_weekend", "hour_sin", "hour_cos", "month_sin",
    "month_cos", "Demand_lag1", "Demand_lag60", "Demand_lag1440",
]

# Use TARGET_COL for data extraction, assuming 'Demand_kW' in the original code
# was meant to refer to the target variable 'use [kW]'.
df_hist = df.dropna(subset=[TARGET_COL]).sort_values('time').reset_index(drop=True)
hist_times = df_hist['time'].values  # for fast nearest-time lookup


def nearest_row(target_time):
    pos = int(np.abs(hist_times - np.datetime64(target_time)).argmin())
    return df_hist.iloc[pos]


def xgb_recursive_forecast(anchor_time, hours=24):
    anchor_row = nearest_row(anchor_time)
    # Use TARGET_COL instead of 'Demand_kW'
    anchor_demand = float(anchor_row[TARGET_COL])
    future_times = pd.date_range(start=anchor_time + pd.Timedelta(hours=1), periods=hours, freq='h')

    preds = []
    last = anchor_demand
    for ft in future_times:
        ref_row = nearest_row(ft - pd.Timedelta(hours=24))
        r = {f: ref_row.get(f, np.nan) for f in XGB_FEATURES if not f.startswith('Demand_lag')}
        # Calendar features are computed dynamically
        h = ft.hour
        m = ft.month
        r['is_weekend'] = float(ft.dayofweek >= 5)
        r['hour_sin'] = float(np.sin(2 * np.pi * h / 24))
        r['hour_cos'] = float(np.cos(2 * np.pi * h / 24))
        r['month_sin'] = float(np.sin(2 * np.pi * m / 12))
        r['month_cos'] = float(np.cos(2 * np.pi * m / 12))
        r['Demand_lag1'] = last
        r['Demand_lag60'] = last
        # Use TARGET_COL instead of 'Demand_kW' for ref_row value
        r['Demand_lag1440'] = float(ref_row.get(TARGET_COL, anchor_demand))

        X = pd.DataFrame([r])[XGB_FEATURES].apply(pd.to_numeric, errors='coerce')
        p = float(xgb_pipeline.predict(X)[0])
        preds.append(p)
        last = p

    return np.array(preds), future_times


# quick smoke test
_p, _t = xgb_recursive_forecast(valid_start)
print(f"Smoke test OK — first 3 predictions: {_p[:3]}")

## 🔮 Step 7: PatchTSMixer Native 24-Hour Forecast

In [ ]:
def patchtsmixer_forecast(anchor_time, hours=24):
    context = df_hourly.loc[:anchor_time].tail(CONTEXT_LENGTH)
    if len(context) < CONTEXT_LENGTH:
        return None, None

    context_vals = context[PATCH_FEATURE_COLS].values  # (96, n_channels)
    scaled = patch_scaler.transform(context_vals)
    x = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        out = patch_model(past_values=x)

    preds_raw = out.prediction_outputs.squeeze(0).cpu().numpy()  # (24, ?) — width

    if preds_raw.shape[-1] == 1:
        target_mean = patch_scaler.mean_[TARGET_CHANNEL_IDX]
        target_scale = patch_scaler.scale_[TARGET_CHANNEL_IDX]
        preds_target = preds_raw.flatten() * target_scale + target_mean
    else:
        # Full channel width returned (matches the documented default behavior):
        # inverse-transform normally, then slice out the target channel.
        preds_inv = patch_scaler.inverse_transform(preds_raw)
        preds_target = preds_inv[:, TARGET_CHANNEL_IDX]

    future_times = pd.date_range(start=anchor_time + pd.Timedelta(hours=1), periods=hours, freq='h')
    return preds_target, future_times


# quick smoke test
_p, _t = patchtsmixer_forecast(valid_start)
print(f"Smoke test OK — first 3 predictions: {_p[:3]}" if _p is not None else "Not enough context at this anchor")


## 📊 Step 8: Run the Fair Comparison Across Many Anchor
 PointsAnchors are taken every 6 hours across the test period (a denser stride would be morethorough but slower — 6-hourly still gives hundreds of independent 24-hour forecasts toaverage over).

In [ ]:
ANCHOR_STRIDE_HOURS = 6

anchors = pd.date_range(start=valid_start, end=valid_end, freq=f'{ANCHOR_STRIDE_HOURS}h')
print(f"Evaluating {len(anchors)} anchor points...")

xgb_preds_all, patch_preds_all, actuals_all = [], [], []

for i, anchor in enumerate(anchors):
    future_times = pd.date_range(start=anchor + pd.Timedelta(hours=1), periods=24, freq='h')
    actual = df_hourly.reindex(future_times)[TARGET_COL].values
    if np.isnan(actual).any():
        continue

    xgb_p, _ = xgb_recursive_forecast(anchor)
    patch_p, _ = patchtsmixer_forecast(anchor)
    if patch_p is None:
        continue

    xgb_preds_all.append(xgb_p)
    patch_preds_all.append(patch_p)
    actuals_all.append(actual)

    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(anchors)} anchors processed...")

xgb_preds_all = np.array(xgb_preds_all)      # (n_anchors, 24)
patch_preds_all = np.array(patch_preds_all)  # (n_anchors, 24)
actuals_all = np.array(actuals_all)          # (n_anchors, 24)

print(f"\n✅ Completed: {len(actuals_all)} valid anchors evaluated.")


## 📈 Step 9: Compute Fair, Full-Horizon Metrics

In [ ]:
def summarize(name, preds, actuals):
    mae = mean_absolute_error(actuals.flatten(), preds.flatten())
    rmse = np.sqrt(mean_squared_error(actuals.flatten(), preds.flatten()))
    r2 = r2_score(actuals.flatten(), preds.flatten())
    per_step_mae = np.abs(actuals - preds).mean(axis=0)
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2': r2, 'per_step_mae': per_step_mae}


xgb_summary = summarize('XGBoost (recursive, 24h)', xgb_preds_all, actuals_all)
patch_summary = summarize('PatchTSMixer (native, 24h)', patch_preds_all, actuals_all)

print("=" * 60)
print("FAIR 24-HOUR COMPARISON — same anchors, same ground truth")
print("=" * 60)
for s in [xgb_summary, patch_summary]:
    print(f"{s['model']:30s} MAE={s['mae']:.4f}  RMSE={s['rmse']:.4f}  R2={s['r2']:.4f}")
print("=" * 60)

print("\nMAE by forecast step (hours ahead):")
print(f"{'Hour':>6s} | {'XGBoost':>10s} | {'PatchTSMixer':>13s}")
for h in range(24):
    print(f"  +{h+1:2d}h | {xgb_summary['per_step_mae'][h]:10.4f} | {patch_summary['per_step_mae'][h]:13.4f}")


## 📉 Step 10: Plot the Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
hours_axis = np.arange(1, 25)
ax.plot(hours_axis, xgb_summary['per_step_mae'], marker='o', label='XGBoost (recursive)', color='#2563EB')
ax.plot(hours_axis, patch_summary['per_step_mae'], marker='s', label='PatchTSMixer (native)', color='#DC2626')
ax.set_xlabel('Forecast horizon (hours ahead)')
ax.set_ylabel('MAE (kW)')
ax.set_title('Fair 24-Hour-Ahead Comparison: XGBoost (recursive) vs. PatchTSMixer (native)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('fair_24h_comparison.png', dpi=200)
plt.show()


## 💾 Step 11: Save the Fair Comparison ResultsSaved
separately from `model_results.csv` (which holds each model's own native-taskmetrics) since these numbers reflect a specifically constructed, matched 24-hourevaluation and should not be silently merged with the single-step Bi-LSTM/XGBoost numbersalready in that file.

In [ ]:
import os

FAIR_RESULTS_PATH = '/content/drive/MyDrive/EnerVision_AI/Models/model_results_24h_fair.csv'

fair_df = pd.DataFrame([
    {'model': 'XGBoost', 'mae_24h': xgb_summary['mae'], 'rmse_24h': xgb_summary['rmse'], 'r2_24h': xgb_summary['r2'], 'n_anchors': len(actuals_all)},
    {'model': 'PatchTSMixer', 'mae_24h': patch_summary['mae'], 'rmse_24h': patch_summary['rmse'], 'r2_24h': patch_summary['r2'], 'n_anchors': len(actuals_all)},
])

os.makedirs(os.path.dirname(FAIR_RESULTS_PATH), exist_ok=True)
fair_df.to_csv(FAIR_RESULTS_PATH, index=False)
print(f"✅ Saved fair 24-hour comparison to:\n{FAIR_RESULTS_PATH}")
print(fair_df)
